In [ ]:
from ortools.sat.python import cp_model
from datetime import date, timedelta
import calendar
from holidayskr import year_holidays
NAME_MAP = {
    "A": "최영철",
    "B": "홍진우",
    "C": "김다영",
    "D": "강승민",
    "E": "문승환",
    "F": "라영일",
    "G": "이병욱",
    "H": "김동명",
    "I": "김선우",
}
A_ID = "A"
STAFF_IDS = ["B", "C", "D", "E", "F", "G", "H", "I"]
ALL_IDS = [A_ID] + STAFF_IDS

YEAR = 2026
MONTH = 3

REQ_D_TOTAL = 2
REQ_E_TOTAL = 2
N_REQ_TOTAL = []
EXTRA_RED_DATES = []
USE_MIN_OFF_MINUS_1 = True

FORCED_OFF = {
    # "홍진우": ["2026-03-10", "2026-03-11"],
    # "김다영": [3],
}

FORBIDDEN_SHIFTS = {
    # "김다영": {"2026-03-20": ["N"]},
    # "홍진우": {15: ["E"]},
}

PREFERRED_SHIFTS = {
    # "홍진우": {"2026-03-05": ["D"]},
    # "김동명": {21: ["-"]},
}

PAIR_RULE_HARD = False
NEWBIE_TO_MENTORS = {
    "김선우": ["홍진우", "문승환"],
}

PREFER_TOGETHER = [
    # ("홍진우", "강승민"),
]
AVOID_TOGETHER = [
    # ("홍진우", "김다영"),
]

W_PREF_SHIFT = 120
W_PAIR_MISS = 500
W_TOGETHER_BONUS = 60
W_AVOID_PENALTY = 120

def days_in_month(year: int, month: int) -> int:
    return calendar.monthrange(year, month)[1]

def weekday_kor(dt: date) -> str:
    kor = ["월", "화", "수", "목", "금", "토", "일"]
    return kor[dt.weekday()]

def is_sunday(dt: date) -> bool:
    return dt.weekday() == 6

def is_weekend(dt: date) -> bool:
    return dt.weekday() in (5, 6)

NUM_DAYS = days_in_month(YEAR, MONTH)
START_DATE = date(YEAR, MONTH, 1)
days = range(NUM_DAYS)

def parse_day_key(k):
    if isinstance(k, int):
        if not (1 <= k <= NUM_DAYS):
            raise ValueError(f"일자 {k}가 범위를 벗어남(1~{NUM_DAYS})")
        return k - 1
    if isinstance(k, str):
        dt = date.fromisoformat(k)
        if dt.year != YEAR or dt.month != MONTH:
            raise ValueError(f"날짜 {k}가 {YEAR}-{MONTH:02d}에 속하지 않음")
        return dt.day - 1
    raise TypeError("키는 'YYYY-MM-DD' 또는 일자(int)만 허용")

def get_kr_holidays_in_month(year: int, month: int) -> dict[date, str]:
    out: dict[date, str] = {}
    for dt, name in year_holidays(str(year)):
        if dt.year == year and dt.month == month:
            out[dt] = name
    return out

#def calc_red_dates(year: int, month: int, extra_red_dates: list[date]):
    last = calendar.monthrange(year, month)[1]
    weekend_set = {date(year, month, d) for d in range(1, last + 1) if is_weekend(date(year, month, d))}
    holiday_name_map = get_kr_holidays_in_month(year, month)
    holiday_set = set(holiday_name_map.keys())
    extra_set = {d for d in extra_red_dates if d.year == year and d.month == month}
    red_dates = weekend_set | holiday_set | extra_set
    return red_dates, holiday_name_map

RED_DATES, HOLIDAY_NAME_MAP = calc_red_dates(YEAR, MONTH, EXTRA_RED_DATES)
MIN_OFF = len(RED_DATES)
MIN_OFF_HARD = max(0, MIN_OFF - 1) if USE_MIN_OFF_MINUS_1 else MIN_OFF
OFF_TARGET = MIN_OFF

print(f"== 빨간날 총 {MIN_OFF}일 | 휴무 하드최소 {MIN_OFF_HARD}일 | 휴무 목표 {OFF_TARGET}일 ==")

TARGET_TOTAL_N = 8 * 6

if not N_REQ_TOTAL:
    N_REQ_TOTAL = [(1 if is_weekend(START_DATE + timedelta(days=d)) else 2) for d in range(NUM_DAYS)]

cur_total_n = sum(N_REQ_TOTAL)
if cur_total_n > TARGET_TOTAL_N:
    candidates = [d for d in range(NUM_DAYS) if (not is_weekend(START_DATE + timedelta(days=d))) and N_REQ_TOTAL[d] == 2]
    need = cur_total_n - TARGET_TOTAL_N
    if need > len(candidates):
        raise SystemExit("[불가능] N 총량을 48로 낮출 후보 부족")
    for d in candidates[:need]:
        N_REQ_TOTAL[d] = 1
elif cur_total_n < TARGET_TOTAL_N:
    candidates = [d for d in range(NUM_DAYS) if (not is_weekend(START_DATE + timedelta(days=d))) and N_REQ_TOTAL[d] == 1]
    need = TARGET_TOTAL_N - cur_total_n
    if need > len(candidates):
        raise SystemExit("[불가능] N 총량을 48로 올릴 후보 부족")
    for d in candidates[:need]:
        N_REQ_TOTAL[d] = 2

assert sum(N_REQ_TOTAL) == TARGET_TOTAL_N
print("== N 총량 ==", sum(N_REQ_TOTAL), "(목표 48)")

SHIFT_D, SHIFT_E, SHIFT_N, SHIFT_OFF = 0, 1, 2, 3
SHIFT_NAMES = {SHIFT_D: "D", SHIFT_E: "E", SHIFT_N: "N", SHIFT_OFF: "-"}
SHIFT_CODE = {"D": SHIFT_D, "E": SHIFT_E, "N": SHIFT_N, "-": SHIFT_OFF}

P_ALL = len(ALL_IDS)
idxA = ALL_IDS.index("A")
idxB = ALL_IDS.index("B")
idxC = ALL_IDS.index("C")

ID_TO_IDX = {pid: i for i, pid in enumerate(ALL_IDS)}
NAME_TO_ID = {v: k for k, v in NAME_MAP.items()}

def name_to_idx(name: str) -> int:
    pid = NAME_TO_ID.get(name)
    if pid is None:
        raise ValueError(f"이름 '{name}'이 NAME_MAP에 없습니다.")
    return ID_TO_IDX[pid]

people_all = range(P_ALL)
people_nonA = [i for i in people_all if i != idxA]
shifts = (SHIFT_D, SHIFT_E, SHIFT_N, SHIFT_OFF)

model = cp_model.CpModel()

x = {(i, d, s): model.NewBoolVar(f"x_{ALL_IDS[i]}_{d}_{s}")
     for i in people_all for d in days for s in shifts}

for i in people_all:
    for d in days:
        model.Add(sum(x[i, d, s] for s in shifts) == 1)

for d in days:
    model.Add(sum(x[i, d, SHIFT_D] for i in people_all) == REQ_D_TOTAL)
    model.Add(sum(x[i, d, SHIFT_E] for i in people_all) == REQ_E_TOTAL)
    model.Add(sum(x[i, d, SHIFT_N] for i in people_all) == N_REQ_TOTAL[d])

for d in days:
    model.Add(x[idxA, d, SHIFT_N] == 0)

for i in people_nonA:
    model.Add(sum(x[i, d, SHIFT_N] for d in days) == 6)

for i in people_nonA:
    for d in range(NUM_DAYS - 2):
        model.Add(x[i, d, SHIFT_N] + x[i, d+1, SHIFT_N] + x[i, d+2, SHIFT_N] <= 2)

for i in people_nonA:
    for d in range(NUM_DAYS - 1):
        model.Add(x[i, d, SHIFT_N] + x[i, d+1, SHIFT_D] <= 1)
        model.Add(x[i, d, SHIFT_N] + x[i, d+1, SHIFT_E] <= 1)

for i in people_nonA:
    for d in range(NUM_DAYS - 2):
        model.Add(x[i, d, SHIFT_N] + x[i, d+2, SHIFT_D] <= 1)

for i in people_nonA:
    for d in range(NUM_DAYS - 1):
        model.Add(x[i, d, SHIFT_E] + x[i, d+1, SHIFT_D] <= 1)

for d in days:
    if N_REQ_TOTAL[d] == 1:
        model.Add(x[idxB, d, SHIFT_N] == 0)
        model.Add(x[idxC, d, SHIFT_N] == 0)
    else:
        model.Add(x[idxB, d, SHIFT_N] + x[idxC, d, SHIFT_N] <= 1)

for name, dom_list in FORCED_OFF.items():
    i = name_to_idx(name)
    for k in dom_list:
        d = parse_day_key(k)
        model.Add(x[i, d, SHIFT_OFF] == 1)

for name, rule in FORBIDDEN_SHIFTS.items():
    i = name_to_idx(name)
    for day_key, codes in rule.items():
        d = parse_day_key(day_key)
        for code in codes:
            if code not in SHIFT_CODE:
                raise ValueError(f"금지근무 코드 오류: {code}")
            model.Add(x[i, d, SHIFT_CODE[code]] == 0)

OFF_cnt, WORK_cnt, work = {}, {}, {}

for i in people_all:
    OFF_cnt[i] = model.NewIntVar(0, NUM_DAYS, f"OFFcnt_{ALL_IDS[i]}")
    model.Add(OFF_cnt[i] == sum(x[i, d, SHIFT_OFF] for d in days))
    model.Add(OFF_cnt[i] >= MIN_OFF_HARD)

    work[i] = {}
    for d in days:
        work[i][d] = model.NewBoolVar(f"work_{ALL_IDS[i]}_{d}")
        model.Add(work[i][d] == 1 - x[i, d, SHIFT_OFF])

    WORK_cnt[i] = model.NewIntVar(0, NUM_DAYS, f"WORKcnt_{ALL_IDS[i]}")
    model.Add(WORK_cnt[i] == sum(work[i][d] for d in days))

for i in people_all:
    for d in range(NUM_DAYS - 5):
        model.Add(sum(work[i][d+k] for k in range(6)) <= 5)

five_consec_flags = []
for i in people_all:
    for d in range(NUM_DAYS - 4):
        f = model.NewBoolVar(f"five_consec_{ALL_IDS[i]}_{d}")
        model.Add(sum(work[i][d+k] for k in range(5)) == 5).OnlyEnforceIf(f)
        model.Add(sum(work[i][d+k] for k in range(5)) <= 4).OnlyEnforceIf(f.Not())
        five_consec_flags.append(f)

OFF_dev = {}
for i in people_all:
    OFF_dev[i] = model.NewIntVar(0, NUM_DAYS, f"OFFdev_{ALL_IDS[i]}")
    diff = model.NewIntVar(-NUM_DAYS, NUM_DAYS, f"OFFdiff_{ALL_IDS[i]}")
    model.Add(diff == OFF_cnt[i] - OFF_TARGET)
    model.AddAbsEquality(OFF_dev[i], diff)

D_cnt, E_cnt = {}, {}
for i in people_nonA:
    D_cnt[i] = model.NewIntVar(0, NUM_DAYS, f"Dcnt_{ALL_IDS[i]}")
    E_cnt[i] = model.NewIntVar(0, NUM_DAYS, f"Ecnt_{ALL_IDS[i]}")
    model.Add(D_cnt[i] == sum(x[i, d, SHIFT_D] for d in days))
    model.Add(E_cnt[i] == sum(x[i, d, SHIFT_E] for d in days))

DE_gap = {}
for i in people_nonA:
    DE_gap[i] = model.NewIntVar(0, NUM_DAYS, f"DEgap_{ALL_IDS[i]}")
    diff = model.NewIntVar(-NUM_DAYS, NUM_DAYS, f"DEdiff_{ALL_IDS[i]}")
    model.Add(diff == D_cnt[i] - E_cnt[i])
    model.AddAbsEquality(DE_gap[i], diff)

maxD = model.NewIntVar(0, NUM_DAYS, "maxD")
minD = model.NewIntVar(0, NUM_DAYS, "minD")
maxE = model.NewIntVar(0, NUM_DAYS, "maxE")
minE = model.NewIntVar(0, NUM_DAYS, "minE")
model.AddMaxEquality(maxD, list(D_cnt.values()))
model.AddMinEquality(minD, list(D_cnt.values()))
model.AddMaxEquality(maxE, list(E_cnt.values()))
model.AddMinEquality(minE, list(E_cnt.values()))

triple_off_flags = []
for i in people_nonA:
    for d in range(NUM_DAYS - 2):
        t = model.NewBoolVar(f"tripleOFF_{ALL_IDS[i]}_{d}")
        model.Add(x[i, d, SHIFT_OFF] + x[i, d+1, SHIFT_OFF] + x[i, d+2, SHIFT_OFF] == 3).OnlyEnforceIf(t)
        model.Add(x[i, d, SHIFT_OFF] + x[i, d+1, SHIFT_OFF] + x[i, d+2, SHIFT_OFF] <= 2).OnlyEnforceIf(t.Not())
        triple_off_flags.append(t)

sunday_indexes = [d for d in days if is_sunday(START_DATE + timedelta(days=d))]
missing_sunD = {}
for i in people_nonA:
    m = model.NewBoolVar(f"missSunD_{ALL_IDS[i]}")
    sum_sunD = sum(x[i, d, SHIFT_D] for d in sunday_indexes)
    model.Add(sum_sunD >= 1).OnlyEnforceIf(m.Not())
    model.Add(sum_sunD == 0).OnlyEnforceIf(m)
    missing_sunD[i] = m

special_days = sorted([(dt - START_DATE).days for dt in RED_DATES])
special_work = {}
for i in people_nonA:
    special_work[i] = model.NewIntVar(0, len(special_days), f"SPW_{ALL_IDS[i]}")
    model.Add(special_work[i] == sum(
        x[i, d, SHIFT_D] + x[i, d, SHIFT_E] + x[i, d, SHIFT_N]
        for d in special_days
    ))
maxSP = model.NewIntVar(0, len(special_days), "maxSP")
minSP = model.NewIntVar(0, len(special_days), "minSP")
model.AddMaxEquality(maxSP, list(special_work.values()))
model.AddMinEquality(minSP, list(special_work.values()))

A_pattern_miss = []
for d in days:
    r = d % 3
    pref = x[idxA, d, SHIFT_E] if r == 0 else (x[idxA, d, SHIFT_OFF] if r == 1 else x[idxA, d, SHIFT_D])
    miss = model.NewBoolVar(f"A_miss_{d}")
    model.Add(miss + pref == 1)
    A_pattern_miss.append(miss)

A_day1_notE = model.NewBoolVar("A_day1_notE")
model.Add(A_day1_notE + x[idxA, 0, SHIFT_E] == 1)

pref_flags = []
for name, rule in PREFERRED_SHIFTS.items():
    i = name_to_idx(name)
    for day_key, codes in rule.items():
        d = parse_day_key(day_key)
        allowed_vars = []
        for code in codes:
            if code not in SHIFT_CODE:
                raise ValueError(f"희망근무 코드 오류: {code}")
            allowed_vars.append(x[i, d, SHIFT_CODE[code]])

        miss = model.NewBoolVar(f"pref_miss_{i}_{d}")
        or_var = model.NewBoolVar(f"pref_or_{i}_{d}")
        for v in allowed_vars:
            model.Add(or_var >= v)
        model.Add(or_var <= sum(allowed_vars))
        model.Add(miss + or_var == 1)
        pref_flags.append(miss)

pair_miss_flags = []
for newbie_name, mentors in NEWBIE_TO_MENTORS.items():
    nb_i = name_to_idx(newbie_name)
    mentor_idx = [name_to_idx(m) for m in mentors]

    for d in days:
        nb_work = model.NewBoolVar(f"nb_work_{nb_i}_{d}")
        model.Add(nb_work == 1 - x[nb_i, d, SHIFT_OFF])

        mentor_work_sum = model.NewIntVar(0, len(mentor_idx), f"mentor_work_sum_{nb_i}_{d}")
        model.Add(mentor_work_sum == sum(1 - x[m_i, d, SHIFT_OFF] for m_i in mentor_idx))

        if PAIR_RULE_HARD:
            model.Add(mentor_work_sum >= 1).OnlyEnforceIf(nb_work)
        else:
            miss = model.NewBoolVar(f"pair_miss_{nb_i}_{d}")
            model.Add(miss <= nb_work)
            zero = model.NewBoolVar(f"mentor_zero_{nb_i}_{d}")
            model.Add(mentor_work_sum == 0).OnlyEnforceIf(zero)
            model.Add(mentor_work_sum >= 1).OnlyEnforceIf(zero.Not())
            model.Add(miss >= nb_work - (1 - zero))
            pair_miss_flags.append(miss)

together_bonus_flags, avoid_penalty_flags = [], []

def both_work_flag(i1, i2, d, tag):
    f = model.NewBoolVar(f"{tag}_{i1}_{i2}_{d}")
    w1 = model.NewBoolVar(f"{tag}_w1_{i1}_{d}")
    w2 = model.NewBoolVar(f"{tag}_w2_{i2}_{d}")
    model.Add(w1 == 1 - x[i1, d, SHIFT_OFF])
    model.Add(w2 == 1 - x[i2, d, SHIFT_OFF])
    model.Add(f <= w1)
    model.Add(f <= w2)
    model.Add(f >= w1 + w2 - 1)
    return f

for n1, n2 in PREFER_TOGETHER:
    i1, i2 = name_to_idx(n1), name_to_idx(n2)
    for d in days:
        together_bonus_flags.append(both_work_flag(i1, i2, d, "together"))

for n1, n2 in AVOID_TOGETHER:
    i1, i2 = name_to_idx(n1), name_to_idx(n2)
    for d in days:
        avoid_penalty_flags.append(both_work_flag(i1, i2, d, "avoid"))

W_OFF_TARGET = 250
W_A_PATTERN_MISS = 80
W_A_DAY1_E = 200

W_FIVE_CONSEC = 400
W_TRIPLE_OFF = 200
W_MISS_SUNDAY_D = 500

W_DE_GAP = 200
W_FAIR_D = 120
W_FAIR_E = 120
W_SPECIAL_FAIR = 20

model.Minimize(
    W_OFF_TARGET * sum(OFF_dev.values())
    + W_A_PATTERN_MISS * sum(A_pattern_miss)
    + W_A_DAY1_E * A_day1_notE
    + W_FIVE_CONSEC * sum(five_consec_flags)
    + W_TRIPLE_OFF * sum(triple_off_flags)
    + W_MISS_SUNDAY_D * sum(missing_sunD.values())
    + W_DE_GAP * sum(DE_gap.values())
    + W_FAIR_D * (maxD - minD)
    + W_FAIR_E * (maxE - minE)
    + W_SPECIAL_FAIR * (maxSP - minSP)
    + W_PREF_SHIFT * sum(pref_flags)
    + W_PAIR_MISS * sum(pair_miss_flags)
    - W_TOGETHER_BONUS * sum(together_bonus_flags)
    + W_AVOID_PENALTY * sum(avoid_penalty_flags))

solver = cp_model.CpSolver()
solver.parameters.max_time_in_seconds = 30.0
solver.parameters.num_search_workers = 8

status = solver.Solve(model)
if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    print("\n답을 찾지 못했습니다.")
    raise SystemExit

def get_row_shifts(i: int) -> list[str]:
    row = []
    for d in range(NUM_DAYS):
        assigned = "?"
        for s in shifts:
            if solver.Value(x[i, d, s]) == 1:
                assigned = SHIFT_NAMES[s]
                break
        row.append(assigned)
    return row

date_headers = [f"{(START_DATE + timedelta(days=d)).day:02d}({weekday_kor(START_DATE + timedelta(days=d))})"
                for d in range(NUM_DAYS)]

print("\n== 월간 근무표 (이름 행 / 날짜 열) ==")
print("\t".join(["이름"] + date_headers + ["D", "E", "N", "근무", "휴무"]))

rows_by_name = {}
for i, pid in enumerate(ALL_IDS):
    name = NAME_MAP[pid]
    row = get_row_shifts(i)
    rows_by_name[name] = row

    Dn = row.count("D")
    En = row.count("E")
    Nn = row.count("N")
    Wk = Dn + En + Nn
    Of = row.count("-")

    print("\t".join([name] + row + [str(Dn), str(En), str(Nn), str(Wk), str(Of)]))

== 빨간날 총 10일 | 휴무 하드최소 9일 | 휴무 목표 10일 ==
== N 총량 == 48 (목표 48)

== 월간 근무표 (이름 행 / 날짜 열) ==
이름	01(일)	02(월)	03(화)	04(수)	05(목)	06(금)	07(토)	08(일)	09(월)	10(화)	11(수)	12(목)	13(금)	14(토)	15(일)	16(월)	17(화)	18(수)	19(목)	20(금)	21(토)	22(일)	23(월)	24(화)	25(수)	26(목)	27(금)	28(토)	29(일)	30(월)	31(화)	D	E	N	근무	휴무
최영철	E	-	D	-	-	D	E	-	D	E	-	D	E	-	D	E	-	D	E	-	D	E	-	D	E	-	D	E	-	D	E	10	10	0	20	11
홍진우	D	-	-	D	D	-	-	E	N	-	N	N	-	E	-	N	-	E	E	E	E	-	D	D	-	D	-	D	E	N	N	7	7	6	20	11
김다영	E	E	E	-	D	-	-	D	-	D	-	D	D	D	-	E	N	N	-	N	-	E	E	N	N	-	N	-	E	-	D	7	7	6	20	11
강승민	-	D	-	N	-	-	D	D	-	D	D	N	N	-	E	-	E	E	N	N	-	-	E	E	E	-	D	-	N	-	-	6	6	6	18	13
문승환	-	N	N	-	E	E	N	-	E	N	-	-	D	E	-	D	D	D	-	D	-	D	N	N	-	E	E	-	D	-	E	7	7	6	20	11
라영일	-	-	E	E	-	D	D	-	N	-	E	E	N	-	N	N	-	-	N	-	E	-	D	-	D	E	-	D	D	N	-	6	6	6	18	13
이병욱	-	D	D	-	E	N	-	N	-	N	N	-	-	D	E	-	E	-	D	D	-	D	-	E	N	N	-	E	-	E	-	6	6	6	18	13
김동명	N	-	-	D	-	E	E	-	D	E	E	-	E	-	D	D	N	N	-	-	D	N	-	-	D	D	E	N	-	E	N	7	7	6	20	11
김선우	D	E	-	E	N	-	-	E	E	-	D	E	-	N	-	-	D	-	D	E	N	-	N	-	-	N	N	-	-	D	D	6	6	6	18	13
